In [0]:
course_df = spark.table("lms_analytics.gold_course_performance")
learner_df = spark.table("lms_analytics.gold_learner_performance")
enrollment_df = spark.table("lms_analytics.gold_enrollment_analytics")
activity_df = spark.table("lms_analytics.gold_learning_activity")

print("All Gold tables loaded successfully")

In [0]:
from pyspark.sql.functions import (
    count,
    avg,
    sum,
    round,
    col
)

total_courses = course_df.select("course_id").distinct().count()
total_learners = learner_df.select("learner_id").distinct().count()

total_enrollments = enrollment_df.select(
    sum("total_enrolments")
).collect()[0][0]

avg_progress = learner_df.select(
    round(avg("avg_progress_pct"), 2)
).collect()[0][0]

print("LMS OVERVIEW")
print("Total Courses:", total_courses)
print("Total Learners:", total_learners)
print("Total Enrollments:", total_enrollments)
print("Average Learner Progress:", avg_progress, "%")

In [0]:
course_dashboard_df = course_df.select(
    "course_id",
    "course_title",
    "category",
    "instructor_name",
    "total_enrolments",
    "avg_progress_pct",
    "completion_rate_pct",
    "performance_category"
).orderBy(
    col("completion_rate_pct").desc()
)

display(course_dashboard_df)

In [0]:
top_courses_df = course_df.select(
    "course_id",
    "course_title",
    "category",
    "total_enrolments",
    "completion_rate_pct",
    "avg_progress_pct"
).orderBy(
    col("total_enrolments").desc()
).limit(10)

display(top_courses_df)

In [0]:
learner_dashboard_df = learner_df.select(
    "learner_id",
    "learner_name",
    "city",
    "subscription_type",
    "total_courses",
    "avg_progress_pct",
    "completed_courses",
    "completion_rate_pct",
    "performance_category"
).orderBy(
    col("completion_rate_pct").desc()
)

display(learner_dashboard_df)

In [0]:
from pyspark.sql.functions import sum, count, round, avg

category_analytics_df = enrollment_df.groupBy(
    "category"
).agg(
    sum("total_enrolments").alias("total_enrolments"),
    sum("unique_learners").alias("unique_learners"),
    round(avg("avg_progress_pct"), 2).alias("avg_progress_pct"),
    round(avg("completion_rate_pct"), 2).alias("avg_completion_rate_pct")
).orderBy(
    "total_enrolments",
    ascending=False
)

display(category_analytics_df)

In [0]:
subscription_analytics_df = learner_df.groupBy(
    "subscription_type"
).agg(
    count("learner_id").alias("total_learners"),
    round(avg("avg_progress_pct"), 2).alias("avg_progress_pct"),
    round(avg("completion_rate_pct"), 2).alias("avg_completion_rate_pct")
).orderBy(
    "total_learners",
    ascending=False
)

display(subscription_analytics_df)

In [0]:
activity_status_df = activity_df.groupBy(
    "activity_status"
).agg(
    count("learner_id").alias("total_learners"),
    round(avg("avg_progress_pct"), 2).alias("avg_progress_pct")
).orderBy(
    "total_learners",
    ascending=False
)

display(activity_status_df)

In [0]:
risk_analytics_df = activity_df.groupBy(
    "learner_risk"
).agg(
    count("learner_id").alias("total_learners"),
    round(avg("avg_progress_pct"), 2).alias("avg_progress_pct")
).orderBy(
    "total_learners",
    ascending=False
)

display(risk_analytics_df)

In [0]:
at_risk_learners_df = activity_df.filter(
    col("learner_risk").isin("High Risk", "Medium Risk")
).select(
    "learner_id",
    "learner_name",
    "total_enrolments",
    "avg_progress_pct",
    "last_activity_date",
    "days_since_last_activity",
    "activity_status",
    "learner_risk"
).orderBy(
    col("days_since_last_activity").desc(),
    col("avg_progress_pct").asc()
).limit(10)

display(at_risk_learners_df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

kpi_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", DoubleType(), False)
])

kpi_summary_df = spark.createDataFrame(
    [
        ("Total Courses", float(total_courses)),
        ("Total Learners", float(total_learners)),
        ("Total Enrollments", float(total_enrollments)),
        ("Average Learner Progress", float(avg_progress))
    ],
    schema=kpi_schema
)

display(kpi_summary_df)

In [0]:
course_completion_chart = course_df.select(
    "course_title",
    "category",
    "completion_rate_pct"
).orderBy(
    col("completion_rate_pct").desc()
)

display(course_completion_chart)

Databricks visualization. Run in Databricks to view.

In [0]:
category_chart = category_analytics_df.select(
    "category",
    "total_enrolments"
).orderBy(
    col("total_enrolments").desc()
)

display(category_chart)

Databricks visualization. Run in Databricks to view.

In [0]:
risk_chart = risk_analytics_df.select(
    "learner_risk",
    "total_learners"
)

display(risk_chart)

Databricks visualization. Run in Databricks to view.

In [0]:
top_courses_chart = course_df.select(
    "course_title",
    "total_enrolments"
).orderBy(
    col("total_enrolments").desc()
).limit(10)

display(top_courses_chart)

Databricks visualization. Run in Databricks to view.